# 失败/边界场景检查

四个 case，一个 case 一个 cell。每个 cell 采 30 帧，
记录 `f/quads/u/MLP/置信度/空间决策/当前模式`，**不写 MODE_SEL**。

重点看：遮镜头/过曝时 `confident=0` 或 `decision=-1`，
而不是乱切模式。最后用 `robust.summary_all()` 生成表。


In [ ]:
import os, sys
ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT) and not os.path.isfile(os.path.join(ROOT, 'lib', 'video_setup.py')):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, 'tools'))
import robustness_notebook as robust
print('ROOT =', ROOT)
print('euid =', os.geteuid())


In [ ]:
ol, regs, vdma0, vdma1 = robust.setup(skip_camera=False, use_manual=True)


## Preview

摆场景时反复跑这一格。


In [ ]:
robust.preview(regs)


## Case 0：遮镜头/全黑

期望：`confident=0` 或 `decision=-1`；不要因为全黑硬切 mode 0。


In [ ]:
r0 = robust.probe(regs, 0, n=30, wait=5)


## Case 1：均匀白墙

期望：`u` 低，信任 MLP；不要出现空间覆盖。


In [ ]:
r1 = robust.probe(regs, 1, n=30, wait=5)


## Case 2：过曝

让窗/灯直射或提高增益，观察门限/判决。


In [ ]:
r2 = robust.probe(regs, 2, n=30, wait=5)


## 快速切换（可选）

跑 20 秒实时调度，期间在窗边/正常/暗角之间快速移动镜头；
看切换次数和有没有振荡。


In [ ]:
import live_scheduler_notebook as live
res = live.run_scene(regs, 3, duration=20, wait=5)


## 汇总


In [ ]:
robust.summary_all()


## 收尾


In [ ]:
robust.cleanup(ol, vdma0, vdma1)
